# SGDC — Segmentation-Guided Density Control (Colab)

Trains **Component A** (boundary-aware densification) vs. baseline 3DGS on a Replica scene.

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator -> GPU` (T4/L4/A100).

Repo: branch `sgdc` of `satyaba/gaussian-splatting`. See `docs/sgdc_spec.md` from cell 2 onward.


## 0. Environment check

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda,
      "| capability", torch.cuda.get_device_capability())


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Get the code (branch `sgdc`) + build CUDA submodules

In [ ]:
import os
REPO = "/content/gaussian-splatting"
BRANCH = "sgdc"
FORK = "https://github.com/satyaba/gaussian-splatting.git"

if not os.path.isdir(REPO):
    !git clone {FORK} {REPO}
%cd {REPO}
!git fetch origin
!git checkout {BRANCH}
!git pull
!git submodule update --init --recursive


In [ ]:
import torch
c = torch.cuda.get_device_capability()
ARCH = f"{c[0]}.{c[1]}"
print("TORCH_CUDA_ARCH_LIST =", ARCH)
# Fix missing <cstdint> on CUDA 12.9 / GCC 14 / py3.13
!bash scripts/patch_cuda_submodules.sh


In [ ]:
%cd /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ninja
!rm -rf build && TORCH_CUDA_ARCH_LIST={ARCH} pip install . 2>&1 | tail -n 6

%cd /content/gaussian-splatting/submodules/simple-knn
!rm -rf build && TORCH_CUDA_ARCH_LIST={ARCH} pip install . 2>&1 | tail -n 6

%cd /content/gaussian-splatting/submodules/fused-ssim
!rm -rf build && TORCH_CUDA_ARCH_LIST={ARCH} pip install . 2>&1 | tail -n 6


In [ ]:
%cd /content/gaussian-splatting
import torch
from diff_gaussian_rasterization import GaussianRasterizer
import simple_knn
print("build OK |", torch.cuda.get_device_name(0))


## 2. Dataset

In [ ]:
DATASET = "/content/drive/MyDrive/Magister/Thesis/Script/Dataset/office_0/Sequence_1"
OUT = "/content/output"
import os
os.makedirs(OUT, exist_ok=True)
assert os.path.isdir(DATASET), f"not found: {DATASET}"
print(sorted(os.listdir(DATASET)))


### (optional) Label/image alignment check

Red boundaries should trace object edges in the RGB frames.


In [ ]:
%cd /content/gaussian-splatting
!python -m sgdc.verify_alignment --dataset_dir "{DATASET}" --out /content/sgdc_align_check
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("/content/sgdc_align_check/*.png"))[:3]:
    display(Image(f))


## 3. Smoke test (fast)

In [ ]:
%cd /content/gaussian-splatting
!python train.py -s "{DATASET}" -m {OUT}/smoke --iterations 1500 --disable_viewer \
  --sgdc_enable --f_b 0.5 --f_int 2.0 --eta 0.3


## 4. Full runs (30k): baseline vs SGDC-A

Both runs are deterministic (`safe_state` seeds 0). ~10 min each at ~50 it/s.


In [ ]:
import subprocess, time

def train(tag, extra="", iters=30000):
    log = f"{OUT}/{tag}.log"
    cmd = (f'python train.py -s "{DATASET}" -m {OUT}/{tag} --disable_viewer '
           f'--test_iterations {iters} --save_iterations {iters} {extra}')
    print(">>", cmd)
    t0 = time.time()
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, shell=True, cwd="/content/gaussian-splatting",
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line); f.flush()
            if "Evaluating" in line or "SGDC-diag" in line:
                print(line.rstrip())
        p.wait()
    print(f"{tag}: {time.time()-t0:.0f}s")

train("base")
train("sgdc_a", "--sgdc_enable --f_b 0.5 --f_int 2.0 --eta 0.3 --sgdc_dt_norm 100")


## 5. Metrics

Hypothesis: `sgdc_a` should match/beat `base` PSNR with **fewer Gaussians**.


In [ ]:
import re, glob
from plyfile import PlyData

def psnr_of(log):
    txt = open(log).read()
    m = re.findall(r"Evaluating train: L1 [\d.]+ PSNR ([\d.]+)", txt)
    return max(map(float, m)) if m else float("nan")

def count(tag):
    f = glob.glob(f"{OUT}/{tag}/point_cloud/iteration_30000/point_cloud.ply")
    return len(PlyData.read(f[0]).elements[0]) if f else -1

print(f"{'tag':10s} {'PSNR':>8s} {'#gauss':>10s} {'PSNR/kG':>9s}")
for tag in ["base", "sgdc_a"]:
    p = psnr_of(f"{OUT}/{tag}.log"); n = count(tag)
    print(f"{tag:10s} {p:8.3f} {n:10,d} {p/(n/1000):9.4f}")


### Optional: sweep `f_b` / `f_int` if A underperforms

Stronger suppression of interior over-densification:


In [ ]:
# for fb, fi in [(0.3, 3.0), (0.5, 2.0), (0.5, 4.0)]:
#     train(f"sgdc_a_fb{fb}_fi{fi}",
#           f"--sgdc_enable --f_b {fb} --f_int {fi} --eta 0.3 --sgdc_dt_norm 60")


## 6. Save outputs to Drive

In [ ]:
!mkdir -p "/content/drive/MyDrive/Magister/Thesis/Script/sgdc_output"
!cp -r {OUT}/* "/content/drive/MyDrive/Magister/Thesis/Script/sgdc_output/"
print("copied to Drive")


## Notes / next steps

- **Diagnostic:** `[SGDC-diag]` prints the boundary-distance spread at the first refines. `dt_raw p90` guides `--sgdc_dt_norm`.
- If A earns its keep, next up is **Component C** (cross-boundary `erfc` scale regularizer) and then **B** (per-segment budgets).
- Full method/math: `docs/sgdc_spec.md`.
